In [2]:
from ultralytics import YOLO
import pandas as pd
import os
import glob
import cv2

In [ ]:
run_dirs = sorted(glob.glob("runs_chatbot/task4_train*"))
latest_run_dir = run_dirs[-1]

best_pt = os.path.join(latest_run_dir, "weights", "best.pt")

model = YOLO(best_pt)

metrics = model.val(split="val", verbose=False)

names = metrics.names
precision = metrics.box.p
recall = metrics.box.r
f1 = metrics.box.f1
ap50 = metrics.box.ap50
ap = metrics.box.ap

print("\n\nPer-class metrics:")
for i, name in names.items():
    print(f"Class: {names[i]}")
    print(f"Precision: {round(precision[i], 3)}")
    print(f"Recall: {round(recall[i], 3)}")
    print(f"F1: {round(f1[i], 3)}")
    print(f"AP@0.5: {round(ap50[i], 3)}")
    print(f"AP@0.5:0.95: {round(ap[i], 3)}\n")

total = 0
for value in f1:
    total += value
mean_f1 = total / len(f1)

print("\nOverall metrics:")
print(f"mAP@0.5     : {round(metrics.box.map50, 3)}")
print(f"mAP@0.5:0.95: {round(metrics.box.map, 3)}")
print(f"Mean Precision : {round(metrics.box.mp, 3)}")
print(f"Mean Recall    : {round(metrics.box.mr, 3)}")
print(f"Mean F1-score  : {round(mean_f1, 3)}")

Ultralytics 8.3.241  Python-3.11.14 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
YOLOv12n summary (fused): 159 layers, 2,557,508 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 489.4174.3 MB/s, size: 37.2 KB)
val: Scanning D:\Daniel\.Mcast Stuffs\Level 6\3rd year\IPCV\Home Assignment\Code\chat_bot_detection_912_ai_118_non_ai_v4.yolov12\valid\labels.cache... 202 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 202/202 184.7Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 6.2it/s 2.1s0.1s
                   all        202        203      0.984      0.966       0.99      0.965
Speed: 1.7ms preprocess, 3.7ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to D:\Daniel\.Mcast Stuffs\Level 6\3rd year\IPCV\Home Assignment\Code\runs\detect\val2


Per-class metrics:
Class: chatgpt
Precision: 1.0
Recall: 0.986
F1: 0.993
AP@0.5: 0.995
AP@0.5:0.95: 0.966

Cl

In [ ]:
matrix = metrics.confusion_matrix.matrix
class_names = metrics.names

confusion_matrices = {}

for i in range(len(class_names)):
    class_name = class_names[i]

    true_positives = matrix[i, i]
    false_negatives = matrix[i].sum() - true_positives
    false_positives = matrix[:, i].sum() - true_positives
    true_negatives = matrix.sum() - (true_positives + false_negatives + false_positives)

    df = pd.DataFrame(
        [[true_positives, false_negatives],
         [false_positives, true_negatives]],
        index=["Actual Positive", "Actual Negative"],
        columns=["Predicted Positive", "Predicted Negative"]
    )

    confusion_matrices[class_name] = df

    print(class_name)
    display(df)

chatgpt


,Predicted Positive,Predicted Negative
Actual Positive,62.0,0.0
Actual Negative,1.0,162.0


claude


,Predicted Positive,Predicted Negative
Actual Positive,64.0,22.0
Actual Negative,0.0,139.0


gemini


,Predicted Positive,Predicted Negative
Actual Positive,54.0,2.0
Actual Negative,1.0,168.0


non-chatbot


,Predicted Positive,Predicted Negative
Actual Positive,18.0,1.0
Actual Negative,3.0,203.0


In [4]:
os.makedirs("Annotated_Results", exist_ok=True)

model = YOLO("runs_chatbot/task4_train/weights/best.pt")
print("Model loaded")

class_names = model.names
print("Classes:", class_names)

colors = {
    "chatgpt": (0, 255, 255),
    "claude": (255, 0, 255),
    "gemini": (0, 0, 255),
    "non-chatbot": (0, 255, 0),
}

cap = cv2.VideoCapture("TestMedia/Test_Video.mp4")
if not cap.isOpened():
    raise RuntimeError("Could not open video")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out_path = os.path.join("Annotated_Results", "annotated_video.mp4")
writer = cv2.VideoWriter(out_path, fourcc, fps, (width, height))

print("Processing video...")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    result = model.predict(frame, conf=0.4, verbose=False)[0]
    annotated = frame.copy()

    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])

        label = class_names[cls_id]
        color = colors.get(label, (255, 255, 255))

        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
        cv2.putText(annotated, f"{label} {conf:.2f}", (x1, max(0, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    writer.write(annotated)

cap.release()
writer.release()

print("Done")
print("Saved:", out_path)

Model loaded
Classes: {0: 'chatgpt', 1: 'claude', 2: 'gemini', 3: 'non-chatbot'}
Processing video...
Done
Saved: Annotated_Results\annotated_video.mp4


In [2]:
os.makedirs("Annotated_Results", exist_ok=True)

model = YOLO("runs_chatbot/task4_train/weights/best.pt")
print("Model loaded")

class_names = model.names 
print("Classes:", class_names)

colors = {
    "chatgpt": (0, 255, 255),
    "claude": (255, 0, 255),
    "gemini": (0, 0, 255),
    "non-chatbot": (0, 255, 0),
}

image_files = [f for f in os.listdir("TestMedia") if f.lower().endswith((".jpg", ".png", ".jpeg"))]

for img_name in image_files:
    img_path = os.path.join("TestMedia", img_name)
    image = cv2.imread(img_path)
    
    result = model.predict(image, conf=0.4, verbose=False)[0]
    annotated = image.copy()

    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])

        label = class_names[cls_id]

        color = colors.get(label, (255, 255, 255))

        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
        cv2.putText(annotated, f"{label} {conf:.2f}", (x1, max(0, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    out_path = os.path.join("Annotated_Results", img_name)
    cv2.imwrite(out_path, annotated)
    print("Saved:", out_path)

print("Done")


Model loaded
Classes: {0: 'chatgpt', 1: 'claude', 2: 'gemini', 3: 'non-chatbot'}
Saved: Annotated_Results\hidden_chatgpt.png
Saved: Annotated_Results\hidden_claude.png
Saved: Annotated_Results\hidden_gemini.png
Saved: Annotated_Results\Screenshot 2026-01-20 111138.png
Saved: Annotated_Results\smol_chatgpt.png
Saved: Annotated_Results\smol_claude.png
Saved: Annotated_Results\smol_gemini.png
Done
